# Loading and Inspecting XPCS Data

This notebook demonstrates how to load XPCS HDF5 result files and inspect their contents.

## Prerequisites

- xpcsviewer installed (`pip install xpcsviewer` or `uv sync`)
- An XPCS HDF5 result file (Multitau or Twotime analysis)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Load an HDF5 File

The `XpcsFile` class is the main entry point for working with XPCS data.

In [ ]:
from xpcsviewer.xpcs_file import XpcsFile

# Replace with your actual file path
DATA_FILE = "/path/to/A001_Multitau_result.hdf"

xf = XpcsFile(DATA_FILE)
print(xf)

## 2. Explore the Analysis Type

Each file has an analysis type that determines what data is available.

In [ ]:
print(f"Analysis type: {xf.atype}")
print(f"File label: {xf.label}")
print(f"File path: {xf.fname}")

## 3. Inspect HDF5 Metadata

The `get_hdf_info()` method returns a dictionary of all HDF5 groups and datasets.

In [ ]:
info = xf.get_hdf_info()

# Show top-level keys
if isinstance(info, dict):
    for key in list(info.keys())[:20]:
        print(f"  {key}")

## 4. Extract G2 Correlation Data

The `get_g2_data()` method returns delay times, G2 values, errors, and Q-bin labels.

In [ ]:
q_values, t_el, g2, g2_err, labels = xf.get_g2_data()

print(f"Number of Q bins: {len(q_values)}")
print(f"Number of delay points: {len(t_el)}")
print(f"G2 shape: {g2.shape}")
print(f"G2 error shape: {g2_err.shape}")
print(f"\nFirst 5 Q values: {q_values[:5]}")
print(f"First 5 labels: {labels[:5]}")

## 5. Plot G2 Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot first 6 Q bins
n_plot = min(6, len(q_values))
for i in range(n_plot):
    ax.errorbar(
        t_el, g2[:, i], yerr=g2_err[:, i],
        fmt='o', markersize=3, capsize=2,
        label=labels[i],
    )

ax.set_xscale('log')
ax.set_xlabel('Delay time (s)')
ax.set_ylabel(r'$g_2(\tau)$')
ax.set_title('G2 Autocorrelation Functions')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 6. Q-Range Filtering

Select a subset of Q bins and time range.

In [ ]:
# Select Q range
q_sub, t_sub, g2_sub, g2_err_sub, labels_sub = xf.get_g2_data(
    qrange=(0.01, 0.05),  # Q range in nm^-1
    trange=(0.001, 100),  # Time range in seconds
)

print(f"Filtered: {len(q_sub)} Q bins, {len(t_sub)} time points")

## 7. SAXS 1D Profile

In [ ]:
q, Iq, xlabel, ylabel = xf.get_saxs1d_data()

fig, ax = plt.subplots(figsize=(8, 5))
# Iq may have multiple frames; plot the first
if Iq.ndim == 2:
    ax.loglog(q, Iq[0], '-', linewidth=1)
else:
    ax.loglog(q, Iq, '-', linewidth=1)

ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_title('SAXS 1D Profile')
plt.tight_layout()
plt.show()

## 8. Using the HDF5 Facade (Schema-Validated I/O)

The `HDF5Facade` provides schema-validated reading for stricter data integrity.

In [ ]:
from xpcsviewer.io import HDF5Facade

facade = HDF5Facade()

# Read Q-map with schema validation
try:
    qmap = facade.read_qmap(DATA_FILE)
    print(f"Q-map type: {type(qmap)}")
    if hasattr(qmap, 'sqmap'):
        print(f"Q-map shape: {qmap.sqmap.shape}")
        print(f"Q-map unit: {qmap.sqmap_unit}")
except Exception as e:
    print(f"Q-map reading skipped (may not exist in file): {e}")

## 9. Cleanup

Always close the file when done (or use context managers).

In [ ]:
xf.close()
print("File closed.")

## Summary

| Task | Method |
|------|--------|
| Load file | `XpcsFile(path)` |
| Check type | `xf.atype` |
| Get metadata | `xf.get_hdf_info()` |
| Extract G2 | `xf.get_g2_data(qrange, trange)` |
| Get SAXS | `xf.get_saxs1d_data()` |
| Schema I/O | `HDF5Facade().read_qmap(path)` |

Next: [02_g2_analysis.ipynb](02_g2_analysis.ipynb) for G2 fitting and analysis.